In [1]:
#pip install ultralytics

^C


In [1]:
from ultralytics import YOLO
import cv2
import numpy as np
import pandas as pd
import ipywidgets as widgets
import matplotlib.pyplot as plt
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.image import img_to_array

In [2]:
args ={
    "face_model":"./model/FaceDetectModel/train5/weights/best.pt",
    "emotion_model":"./model/Emotion_models/model_emotion.h5",
    "age_model":"./model/VN_Age_models/model_VN_Age.h5",

    "label_emotion_encoder":"./model/Emotion_models/label_encoder.pkl",
    "emotion_labels":"./model/Emotion_models/emotion_labels.pkl",

    "age_encoder":"./model/VN_Age_models/age_label_encoder.pkl",
    "age_labels":"./model/VN_Age_models/age_range_labels.pkl"
    }

In [3]:
def process_frame_emotion(frame, yolo_model, cnn_model,age_model):
    # Load label encoder và emotion_labels
    import pickle
    with open(args["label_emotion_encoder"], "rb") as f:
        le = pickle.load(f)
    with open(args["emotion_labels"], "rb") as f:
        emotion_labels = pickle.load(f)

    with open(args["age_encoder"], "rb") as f:
        le = pickle.load(f)
    with open(args["age_labels"], "rb") as f:
        age_labels = pickle.load(f)

    #emotion_pred = model.predict(face_array)
    #emotion_label = emotion_labels[np.argmax(emotion_pred)]

    results = yolo_model(frame, verbose=False)[0]
    for box in results.boxes:
        x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
        conf = box.conf[0].item()

        # Cắt khuôn mặt từ ảnh
        face = frame[y1:y2, x1:x2]
        if face.size == 0:
            continue

        # Tiền xử lý ảnh khuôn mặt cho CNN
        face_input_size = (100, 100)
        face_resized = cv2.resize(face, (100, 100))  # width, height
        face_array = img_to_array(face_resized)
        face_array = face_array.astype("float32") / 255.0
        face_array = np.expand_dims(face_array, axis=0)

        # Dự đoán cảm xúc
        emotion_pred = cnn_model.predict(face_array, verbose=0)
        emotion_label = emotion_labels[np.argmax(emotion_pred)]
        emotion_conf = np.max(emotion_pred)

        # Dự đoán tuổi
        age_pred = age_model.predict(face_array, verbose=0)
        age_label = age_labels[np.argmax(age_pred)]
        age_conf = np.max(age_pred)

        # Hiển thị bounding box và nhãn
        label_text = f"{emotion_label}: {emotion_conf:.1%}"
        label_text2= f"{age_label}: {age_conf:.1%}"
        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(frame, label_text, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX,
                    0.8, (0, 255, 0), 2, cv2.LINE_AA)

    return frame

In [5]:
print("[INFO] Đang tải model YOLO và CNN...")
yolo_model = YOLO(args['face_model'])  # Model YOLO để phát hiện khuôn mặt
cnn_model = load_model(args['emotion_model'])  # Model CNN để phân loại cảm xúc
age_model= load_model(args['age_model'])

# ============================
# Khởi động camera và chạy mô hình
# ============================
print("[INFO] Mở webcam...")
webcamera = cv2.VideoCapture(0)

while True:
    success, frame = webcamera.read()
    if not success:
        break

    processed_frame = process_frame_emotion(frame, yolo_model, cnn_model,age_model)

    cv2.imshow("Emotion Detection", processed_frame)
    if cv2.waitKey(1) == ord('q'):
        break

webcamera.release()
cv2.destroyAllWindows()


[INFO] Đang tải model YOLO và CNN...


[INFO] Mở webcam...
